# Airline Passenger Forecasting
### Forecasting Monthly Passengers with the Classic Box-Jenkins Dataset

**Project 27 of 50 — Time Series Forecasting Portfolio**

## Project Overview

The **AirPassengers** dataset records monthly airline passenger totals from January 1949
to December 1960 — 144 observations that have become the canonical benchmark for
time-series teaching.

| Attribute | Detail |
|---|---|
| **Kaggle slug** | `chirag19/air-passengers` |
| **Columns** | `Month`, `#Passengers` |
| **Target** | Monthly passengers (thousands) |
| **Frequency** | Monthly (MS) |
| **Season length** | 12 (annual) |
| **Signature feature** | *Multiplicative* seasonality — amplitude grows with level |
| **TS library** | StatsForecast |

## Learning Objectives

1. Identify multiplicative vs additive seasonality from a plot
2. Apply log-transform to convert multiplicative to additive structure
3. Run AutoARIMA, AutoETS and AutoTheta on a short (144-point) series
4. Discuss why standard ML/FLAML lag-feature models struggle on very short series
5. Interpret ACF/PACF on a trended, seasonal series
6. Understand why MAPE is the preferred metric when values span a wide range

## Problem Statement

Given 132 months of passenger data (1949–1959), forecast the **next 12 months** (1960).
Reproduce and beat the classic Box-Jenkins ARIMA(0,1,1)(0,1,1)[12] result.

## Why This Project Matters

AirPassengers is used in every major TS textbook precisely because it exhibits
both a clear trend *and* multiplicative seasonality — two properties that expose
weaknesses in models that assume additive structure. Mastering this dataset
builds intuition that transfers directly to retail, energy, and healthcare series.

## Dataset Overview

| Column | Type | Description |
|---|---|---|
| `Month` | string (YYYY-MM) | Year-month identifier |
| `#Passengers` | int | International airline passengers (thousands) |

**Key characteristics**:
- 144 observations (Jan 1949 – Dec 1960)
- Strong upward trend
- Multiplicative seasonality: summer peaks are proportionally larger as the series grows
- No missing values
- Source: Box & Jenkins (1976), *Time Series Analysis: Forecasting and Control*

## Dataset Source & License

- **Kaggle**: https://www.kaggle.com/datasets/chirag19/air-passengers
- **Original source**: Box & Jenkins (1976); freely available as an R built-in dataset
- **License**: Public domain / freely redistributable
- **Usage**: educational benchmark dataset

## Environment Setup

In [ ]:
import subprocess, sys
for p in ["kagglehub","pandas","numpy","matplotlib","seaborn","plotly",
          "statsmodels","scikit-learn","lazypredict","flaml","statsforecast"]:
    try:
        __import__(p.split("[")[0].replace("-","_"))
    except ImportError:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",p])
print("All packages ready.")

## Imports

In [ ]:
import warnings, os, pathlib
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from scipy import stats as scipy_stats
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, AutoTheta, SeasonalNaive
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
print("Imports OK.")

## Configuration

In [ ]:
PROJECT     = "Airline Passenger Forecasting"
KAGGLE_SLUG = "chirag19/air-passengers"
FREQ        = "MS"       # month start
SEASON_LEN  = 12         # annual seasonality
HORIZON     = 12         # forecast 12 months ahead
TEST_SIZE   = HORIZON
VAL_SIZE    = HORIZON
RANDOM_SEED = 42
FLAML_SECS  = 120

DATA_DIR = pathlib.Path("data/airpassengers")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Season={SEASON_LEN} months | Horizon={HORIZON} months")

## Kaggle Credential Check

In [ ]:
kaggle_ok = False
for k in ["KAGGLE_USERNAME","KAGGLE_KEY","KAGGLE_API_TOKEN"]:
    if os.environ.get(k):
        print(f"Credential found: {k}")
        kaggle_ok = True
        break
kj = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kj.exists():
    print(f"kaggle.json at {kj}")
    kaggle_ok = True
if not kaggle_ok:
    print("WARNING: No Kaggle credentials found!")
    print("Set KAGGLE_USERNAME+KAGGLE_KEY or place kaggle.json at ~/.kaggle/")
    print("https://www.kaggle.com/settings -> Create New Token")
else:
    print("Credentials verified.")

## Dataset Download

In [ ]:
import kagglehub
try:
    data_path = pathlib.Path(kagglehub.dataset_download("chirag19/air-passengers"))
    print(f"Dataset at: {data_path}")
except Exception as e:
    print(f"kagglehub failed: {e}  — trying kaggle CLI")
    os.system("kaggle datasets download -d chirag19/air-passengers -p data/airpassengers --unzip")
    data_path = DATA_DIR

csvs = list(data_path.rglob("*.csv"))
print(f"CSV files: {[f.name for f in csvs]}") 

## Load & Inspect

In [ ]:
csv_file = csvs[0]
print(f"Loading: {csv_file.name}")

df_raw = pd.read_csv(csv_file)
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print(df_raw.head(6))

In [ ]:
# Parse date and rename columns
# Typical column names: 'Month' and '#Passengers'
date_col  = [c for c in df_raw.columns if "month" in c.lower() or "date" in c.lower()][0]
val_col   = [c for c in df_raw.columns if c not in [date_col]][0]
print(f"Date col: '{date_col}'  |  Value col: '{val_col}'")

df_raw[date_col] = pd.to_datetime(df_raw[date_col])
ts_df = df_raw.rename(columns={date_col:"ds", val_col:"y"}).sort_values("ds").reset_index(drop=True)
ts_df["y"] = pd.to_numeric(ts_df["y"], errors="coerce")

print(f"\nSeries: {len(ts_df)} monthly observations")
print(f"Range : {ts_df['ds'].min().date()} to {ts_df['ds'].max().date()}")
print(f"\nPassenger stats:")
print(ts_df["y"].describe().round(1))

## Data Validation

In [ ]:
print("DATA VALIDATION")
print("="*40)
print(f"Missing values  : {ts_df['y'].isna().sum()}")
print(f"Duplicate months: {ts_df.duplicated('ds').sum()}")
print(f"Negative values : {(ts_df['y'] < 0).sum()}")
print(f"Min passengers  : {ts_df['y'].min():.0f}k")
print(f"Max passengers  : {ts_df['y'].max():.0f}k")

# Expected: exactly 144 monthly rows
full_range = pd.date_range(ts_df["ds"].min(), ts_df["ds"].max(), freq="MS")
expected = len(full_range)
print(f"\nExpected months : {expected}")
print(f"Actual months   : {len(ts_df)}")
print("Gap-free." if len(ts_df) == expected else "GAPS DETECTED!")

## Exploratory Data Analysis

The AirPassengers series has two defining features visible even to a casual observer:
1. **Upward trend** — total passengers roughly tripled from 1949 to 1960
2. **Multiplicative seasonality** — summer peaks are larger in absolute terms as the series grows

We confirm this visually and via decomposition.

In [ ]:
# Raw series
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_df["ds"], y=ts_df["y"],
    mode="lines+markers", name="Passengers (thousands)",
    line=dict(color="#1565C0", width=2), marker=dict(size=4)))
fig.update_layout(title="Monthly Airline Passengers — Jan 1949 to Dec 1960",
    xaxis_title="Month", yaxis_title="Passengers (thousands)",
    template="plotly_white")
fig.show()

In [ ]:
# Log-transformed series to assess multiplicative structure
ts_df["y_log"] = np.log(ts_df["y"])

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(ts_df["ds"], ts_df["y"], color="#1565C0", lw=2)
axes[0].set_title("Original — Multiplicative seasonality (amplitude grows with level)")
axes[0].set_ylabel("Passengers (thousands)")

axes[1].plot(ts_df["ds"], ts_df["y_log"], color="#2E7D32", lw=2)
axes[1].set_title("Log-transformed — Additive structure after transform")
axes[1].set_ylabel("log(Passengers)")
plt.tight_layout()
plt.show()

In [ ]:
# Monthly averages (shows annual pattern)
ts_df["month_name"] = ts_df["ds"].dt.strftime("%b")
ts_df["month_num"]  = ts_df["ds"].dt.month

monthly_avg = ts_df.groupby("month_num")["y"].mean()
month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
monthly_avg.plot(kind="bar", ax=axes[0], color="#1565C0", edgecolor="black", alpha=0.85)
axes[0].set_xticklabels(month_labels, rotation=30)
axes[0].set_title("Mean Passengers by Month (1949–1960)")
axes[0].set_ylabel("Mean Passengers (thousands)")

# Year-over-year overlay
for yr in ts_df["ds"].dt.year.unique():
    sub = ts_df[ts_df["ds"].dt.year == yr]
    axes[1].plot(sub["month_num"].values, sub["y"].values,
                 alpha=0.6, lw=1.5, label=str(yr))
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(month_labels, rotation=30)
axes[1].set_title("Year-over-Year Comparison")
axes[1].set_ylabel("Passengers (thousands)")
axes[1].legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

ts_df = ts_df.drop(columns=["month_name","month_num"])

In [ ]:
# Seasonal decomposition on LOG series (additive after log = multiplicative original)
ts_log_idx = ts_df.set_index("ds")["y_log"].asfreq("MS")
decomp = seasonal_decompose(ts_log_idx, model="additive", period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0], title="Observed (log)", color="#1565C0")
decomp.trend.plot(ax=axes[1], title="Trend (log)", color="#C62828")
decomp.seasonal.plot(ax=axes[2], title="Seasonal (monthly) — log scale", color="#2E7D32")
decomp.resid.dropna().plot(ax=axes[3], title="Residual (log)", color="#757575", style=".")
plt.tight_layout()
plt.show()

In [ ]:
# ACF / PACF — on log series
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(ts_log_idx.dropna(), lags=36, ax=axes[0],
    title="ACF — log(Passengers)")
plot_pacf(ts_log_idx.dropna(), lags=36, ax=axes[1],
    title="PACF — log(Passengers)")
plt.tight_layout()
plt.show()

In [ ]:
# ADF test on original and log-differenced series
for label, series in [("Original", ts_df["y"]),
                       ("Log", ts_df["y_log"]),
                       ("Log 1st-diff", ts_df["y_log"].diff().dropna())]:
    adf = adfuller(series.dropna())
    verdict = "STATIONARY" if adf[1] < 0.05 else "non-stationary"
    print(f"{label:20s}  p={adf[1]:.4f}  -> {verdict}")

## Target Analysis

In [ ]:
print("Passenger count statistics:")
print(ts_df["y"].describe().round(1))
print(f"\nGrowth factor (last/first): {ts_df['y'].iloc[-1]/ts_df['y'].iloc[0]:.2f}x")
print(f"Seasonality index (max/min monthly avg): "
      f"{ts_df.groupby(ts_df['ds'].dt.month)['y'].mean().max() / ts_df.groupby(ts_df['ds'].dt.month)['y'].mean().min():.2f}x")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ts_df["y"].hist(bins=20, ax=axes[0], edgecolor="black", color="#1565C0", alpha=0.8)
axes[0].set_title("Distribution (right-skewed due to trend)")
scipy_stats.probplot(ts_df["y"], dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot (departure from normality expected)")
plt.tight_layout()
plt.show()

## Train / Validation / Test Split

Temporal split — last 12 months (1960) = test, preceding 12 months (1959) = validation.

With only 144 observations and a 12-month horizon, the split is tight.
This is a realistic scenario for short historical series.

In [ ]:
n = len(ts_df)
ts_train     = ts_df.iloc[:n - TEST_SIZE - VAL_SIZE].copy()
ts_val       = ts_df.iloc[n - TEST_SIZE - VAL_SIZE : n - TEST_SIZE].copy()
ts_test      = ts_df.iloc[n - TEST_SIZE:].copy()
ts_train_val = pd.concat([ts_train, ts_val]).reset_index(drop=True)

assert ts_train["ds"].max() < ts_val["ds"].min()
assert ts_val["ds"].max()   < ts_test["ds"].min()

print(f"Train : {len(ts_train):3d} months  {ts_train['ds'].min().date()} – {ts_train['ds'].max().date()}")
print(f"Val   : {len(ts_val):3d} months  {ts_val['ds'].min().date()} – {ts_val['ds'].max().date()}")
print(f"Test  : {len(ts_test):3d} months  {ts_test['ds'].min().date()} – {ts_test['ds'].max().date()}")

## Preprocessing

For the tabular path (LazyPredict / FLAML) we work on the log-transformed series.
StatsForecast models can handle both original and log scales — we'll fit on the
original and let AutoETS pick the right error model.

No Winsorisation needed — there are no outliers in AirPassengers.

In [ ]:
# Log-transform target for tabular models
ts_train["y_log"]     = np.log(ts_train["y"])
ts_val["y_log"]       = np.log(ts_val["y"])
ts_test["y_log"]      = np.log(ts_test["y"])
ts_train_val["y_log"] = np.log(ts_train_val["y"])
print("Log-transform applied for tabular modeling path.")

## Feature Engineering

In [ ]:
def make_features(df, target_col="y_log"):
    df = df.copy()
    for lag in [1, 2, 3, 6, 12, 24]:
        if lag < len(df):
            df[f"lag_{lag}"] = df[target_col].shift(lag)
    for w in [3, 6, 12]:
        df[f"roll_mean_{w}"] = df[target_col].shift(1).rolling(w).mean()
        df[f"roll_std_{w}"]  = df[target_col].shift(1).rolling(w).std()
    df["month"]   = df["ds"].dt.month
    df["quarter"] = df["ds"].dt.quarter
    df["year"]    = df["ds"].dt.year
    return df

ts_full_log = pd.concat([ts_train, ts_val, ts_test]).reset_index(drop=True)
ts_full_log = ts_full_log.rename(columns={"y":"y_orig"})
ts_full_log["y"] = ts_full_log["y_log"]
ts_feat = make_features(ts_full_log, target_col="y")

FEAT_COLS = [c for c in ts_feat.columns if c not in ["ds","y","y_orig","y_log"]]
n_tr, n_va = len(ts_train), len(ts_val)
feat_train = ts_feat.iloc[:n_tr].dropna().copy()
feat_val   = ts_feat.iloc[n_tr:n_tr+n_va].dropna().copy()
feat_test  = ts_feat.iloc[n_tr+n_va:].dropna().copy()
print(f"Feature cols ({len(FEAT_COLS)}): {FEAT_COLS}")
print(f"Tabular — train:{len(feat_train)} val:{len(feat_val)} test:{len(feat_test)}")

## Baseline Models

- **Naive** — repeat December 1959 for all 12 test months
- **Seasonal Naive (lag-12)** — predict each 1960 month using the same month from 1959
- **12-month Moving Average** — average of the last 12 months

The Seasonal Naive is a surprisingly strong baseline on this series.

In [ ]:
def eval_fc(actual, pred, name):
    a = np.array(actual).flatten()
    p = np.array(pred).flatten()[:len(a)]
    mae  = mean_absolute_error(a, p)
    rmse = float(np.sqrt(mean_squared_error(a, p)))
    mask = a != 0
    mape = float(np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100) if mask.sum() else float("nan")
    print(f"{name:40s}  MAE={mae:7.1f}  RMSE={rmse:7.1f}  MAPE={mape:6.2f}%")
    return dict(model=name, MAE=mae, RMSE=rmse, MAPE=mape)

results = []
actual_test = ts_test["y"].values

# Naive
naive_p = np.full(TEST_SIZE, ts_train_val["y"].iloc[-1])
results.append(eval_fc(actual_test, naive_p, "Naive (last value)"))

# Seasonal naive lag-12
sn12 = ts_train_val["y"].iloc[-SEASON_LEN:].values
sn12 = np.tile(sn12, TEST_SIZE // SEASON_LEN + 1)[:TEST_SIZE]
results.append(eval_fc(actual_test, sn12, "Seasonal Naive (lag-12)"))

# 12-month MA
ma12 = np.full(TEST_SIZE, ts_train_val["y"].iloc[-12:].mean())
results.append(eval_fc(actual_test, ma12, "Moving Average (12-month)"))

print("\nBaselines recorded.")

## LazyPredict Benchmark

**Important caveat**: AirPassengers has only 144 monthly observations.
After lag-feature engineering and dropna(), the effective training set is very small.
LazyPredict results here should be treated as indicative, not definitive.
The dataset is simply too short for reliable tabular ML — this is a key insight.

In [ ]:
X_tr = feat_train[FEAT_COLS]; y_tr = feat_train["y"]
X_va = feat_val[FEAT_COLS];   y_va = feat_val["y"]
print(f"LazyPredict  train:{X_tr.shape}  val:{X_va.shape}")
print("Note: small training set — results are indicative only")
try:
    lz = LazyRegressor(verbose=0, ignore_warnings=True)
    lz_models, _ = lz.fit(X_tr, X_va, y_tr, y_va)
    print("\nTop 15 (on log scale):")
    print(lz_models.head(15).to_string())
    lz_top = lz_models.head(3).index.tolist()
    print(f"\nLazy top-3: {lz_top}")
except Exception as e:
    print(f"LazyPredict error: {e}")
    lz_top = []

## FLAML AutoML

In [ ]:
X_tv = pd.concat([feat_train, feat_val])[FEAT_COLS]
y_tv = pd.concat([feat_train, feat_val])["y"]
X_te = feat_test[FEAT_COLS]

automl = AutoML()
automl.fit(X_tv, y_tv, task="regression", time_budget=FLAML_SECS,
           metric="rmse", verbose=0, seed=RANDOM_SEED)
print(f"Best FLAML model : {automl.best_estimator}")

flaml_pred_log = automl.predict(X_te)
flaml_pred     = np.expm1(flaml_pred_log)   # inverse log-transform
results.append(eval_fc(actual_test, flaml_pred, f"FLAML ({automl.best_estimator}) [exp(log)]"))

print("\nNote: FLAML trained on log-scale; predictions inverse-transformed back to original scale.")

## StatsForecast — AutoARIMA, AutoETS, AutoTheta

**Why StatsForecast on this dataset?**
- AutoARIMA automatically fits the classic Box-Jenkins ARIMA(0,1,1)(0,1,1)[12] or better
- AutoETS picks between ETS(A,A,A) and ETS(M,A,M) — the multiplicative error model
- AutoTheta is a strong competitor on trended series with multiplicative seasonality

We fit on the **original scale** and let the models decide on log/Box-Cox transformations.

In [ ]:
sf_df = ts_train_val[["ds","y"]].copy()
sf_df["unique_id"] = "airpassengers"
sf_df = sf_df[["unique_id","ds","y"]]

sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LEN),
            AutoETS(season_length=SEASON_LEN),
            AutoTheta(season_length=SEASON_LEN),
            SeasonalNaive(season_length=SEASON_LEN)],
    freq="MS", n_jobs=1
)
sf.fit(sf_df)
sf_fc = sf.forecast(h=TEST_SIZE).reset_index(drop=True)

print("StatsForecast forecasts:")
for m in ["AutoARIMA","AutoETS","AutoTheta","SeasonalNaive"]:
    if m in sf_fc.columns:
        results.append(eval_fc(actual_test, sf_fc[m].values, f"SF-{m}"))

In [ ]:
# Plot all forecasts vs actual
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_train_val["ds"], y=ts_train_val["y"],
    name="Train (1949–1959)", line=dict(color="#90CAF9", width=1.5)))
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"],
    name="Actual (1960)", line=dict(color="#0D47A1", width=3)))

palette = ["#F44336","#4CAF50","#FF9800","#9C27B0"]
for i, m in enumerate(["AutoARIMA","AutoETS","AutoTheta","SeasonalNaive"]):
    if m in sf_fc.columns:
        fig.add_trace(go.Scatter(x=ts_test["ds"], y=sf_fc[m].values,
            name=f"SF-{m}", line=dict(width=2, dash="dot", color=palette[i])))

fig.update_layout(title="AirPassengers — Forecasts for Year 1960",
    template="plotly_white", xaxis_title="Month", yaxis_title="Passengers (thousands)",
    legend=dict(orientation="h", y=-0.25))
fig.show()

## Top 3 Model Selection

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print("="*65)
print("ALL MODELS (ranked by RMSE)")
print("="*65)
print(results_df.to_string(index=False))
top3 = results_df.head(3)
print("\nTOP 3:")
print(top3.to_string(index=False))

In [ ]:
fig = px.bar(results_df, x="model", y="RMSE",
    title="Model Comparison — RMSE (lower = better)",
    color="RMSE", color_continuous_scale="RdYlGn_r",
    text=results_df["RMSE"].round(1))
fig.update_layout(xaxis_tickangle=-35, template="plotly_white")
fig.show()

## Final Evaluation of Top 3

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"],
    name="Actual", line=dict(color="#0D47A1", width=3)))

colors3 = ["#F44336","#4CAF50","#FF9800"]
for i, (_, row) in enumerate(top3.iterrows()):
    mname = row["model"]
    pred_vals = None
    if mname.startswith("SF-"):
        key = mname[3:]
        if key in sf_fc.columns:
            pred_vals = sf_fc[key].values
    elif "FLAML" in mname:
        pred_vals = flaml_pred[:TEST_SIZE]
    if pred_vals is not None:
        fig.add_trace(go.Scatter(x=ts_test["ds"], y=pred_vals[:TEST_SIZE],
            name=f"#{i+1} {mname}  RMSE={row['RMSE']:.1f}",
            line=dict(width=2.5, dash="dot", color=colors3[i])))

fig.update_layout(title="Top 3 — Forecast vs Actual (Year 1960)",
    template="plotly_white", xaxis_title="Month", yaxis_title="Passengers (thousands)",
    legend=dict(orientation="h", y=-0.25))
fig.show()

## Error Analysis

In [ ]:
best_name = top3.iloc[0]["model"]
if best_name.startswith("SF-"):
    best_pred = sf_fc[best_name[3:]].values
elif "FLAML" in best_name:
    best_pred = flaml_pred
else:
    best_pred = sn12

errors   = actual_test - best_pred[:TEST_SIZE]
abs_err  = np.abs(errors)
month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]

print(f"Best model     : {best_name}")
print(f"Mean error     : {errors.mean():.1f}k (bias; + = under-forecast)")
print(f"Std errors     : {errors.std():.1f}k")
print(f"Max abs error  : {abs_err.max():.1f}k (month: {month_labels[np.argmax(abs_err)]})")
print(f"Median abs err : {np.median(abs_err):.1f}k")

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].bar(month_labels, errors, color=["#F44336" if e<0 else "#4CAF50" for e in errors],
    edgecolor="black", alpha=0.85)
axes[0].axhline(0, color="black", lw=1)
axes[0].set_title(f"Monthly Errors — {best_name}")
axes[0].set_ylabel("Error (thousands)")

axes[1].scatter(actual_test, best_pred[:TEST_SIZE], s=60, color="#1565C0", alpha=0.8, zorder=3)
mn = min(actual_test.min(), best_pred[:TEST_SIZE].min())
mx = max(actual_test.max(), best_pred[:TEST_SIZE].max())
axes[1].plot([mn,mx],[mn,mx],"r--")
axes[1].set_xlabel("Actual"); axes[1].set_ylabel("Predicted"); axes[1].set_title("Actual vs Predicted")

pct_err = (errors / actual_test) * 100
axes[2].bar(month_labels, pct_err,
    color=["#F44336" if e<0 else "#4CAF50" for e in pct_err], edgecolor="black", alpha=0.85)
axes[2].axhline(0, color="black", lw=1)
axes[2].set_title("Percentage Error by Month"); axes[2].set_ylabel("% Error")
plt.tight_layout()
plt.show()

## Interpretation & Insights

### Key Findings
1. **Multiplicative seasonality requires log-transform or multiplicative error models** —
   models assuming additive structure systematically underforecast summer peaks in later years
2. **AutoETS with multiplicative error** (ETS(M,A,M) or ETS(M,Ad,M)) typically wins on
   this dataset because it explicitly models variance proportional to level
3. **AutoARIMA reproduces Box-Jenkins** — the classic ARIMA(0,1,1)(0,1,1)[12] is usually
   found by AutoARIMA's grid search
4. **Short series limitation** — with only 120 training observations, tabular ML models
   trained on lag features have insufficient data to compete with statistical models
5. **Seasonal Naive is strong** — with clear 12-month cycles, simply repeating last year
   gives a tough baseline; any model must convincingly outperform it

## Limitations

1. **Only 144 observations** — too few for reliable neural or complex ML models
2. **No covariate information** — fuel prices, economic conditions, and route changes
   are not included
3. **Aggregated data** — this is a single world total; individual routes would have
   different patterns
4. **No structural breaks** — the dataset ends before the 1973 oil crisis; real-world
   airline series would include step changes
5. **Educational dataset** — optimised for teaching, not for production forecasting

## How to Improve

1. Fit an explicit ETS(M,A,M) model and compare against AutoETS selection
2. Apply Box-Cox transformation instead of log and tune the lambda parameter
3. Build a state-space model using `statsmodels.tsa.statespace.SARIMAX`
4. Try Facebook Prophet — it handles multiplicative seasonality natively
5. Implement rolling-origin cross-validation with 5 folds

## Production Considerations

1. Monthly airline forecasts are typically generated 3–6 months ahead for schedule planning
2. AutoARIMA requires retraining when new data arrives (monthly cadence)
3. Include confidence intervals for capacity planning — not just point forecasts
4. Anomaly detection: flag months where actual deviates from forecast by >2 MAE

## Common Mistakes

1. **Applying additive decomposition to multiplicative series** — the residuals will show
   heteroscedasticity (variance grows with level)
2. **Ignoring the log-MAPE paradox** — MAPE on the original scale penalises
   under-forecasts more than over-forecasts; use sMAPE for symmetric evaluation
3. **Training ARIMA without differencing identification** — ADF test alone does not
   determine seasonal differencing need; check KPSS and visual inspection
4. **Leaking future months via centered rolling windows** — always use `.shift(1)`
   before any rolling computation

## Mini Challenges

1. **Reproduce Box-Jenkins** — fit `ARIMA(0,1,1)x(0,1,1,12)` manually via statsmodels
   and compare RMSE to AutoARIMA
2. **Additive vs multiplicative** — force AutoETS into additive mode; how much does
   RMSE increase?
3. **Horizon extension** — extend `HORIZON` to 24 months; does MAPE double?
4. **Confidence intervals** — generate 80% and 95% prediction intervals using
   `StatsForecast.forecast_in_sample()` or conformal prediction
5. **Ensemble** — average AutoARIMA and AutoETS predictions; beats either alone?

## Final Summary

### What We Built
- Loaded the classic 144-observation AirPassengers monthly series
- Identified and confirmed multiplicative seasonality via log-transform analysis
- Built Naive, Seasonal Naive, and 12-month MA baselines
- Benchmarked regressors via LazyPredict (noting the short-series limitation)
- Applied FLAML AutoML on log-scale features
- Trained AutoARIMA, AutoETS, AutoTheta, SeasonalNaive via StatsForecast
- Selected top 3 by test-set RMSE and diagnosed monthly error patterns

### Key Takeaways
- Multiplicative seasonality is the defining challenge of this dataset
- Log-transform converts multiplicative to additive structure for compatible models
- AutoETS and AutoARIMA outperform tabular ML because the series is too short for lag-feature methods
- The Seasonal Naive (repeat last year) is harder to beat than it looks

---
*AirPassengers — Public Domain | Box & Jenkins (1976)*
*Educational use*